# Teacher–Student Distillation for Robotics at Scale with Ray on Anyscale

## Introduction

As you scale robotics perception and control systems, you quickly run into a common pattern: you want **large models to learn representations**, but you need **small models to run on robots**.

Teacher–student training is the mechanism that makes this possible. You train or obtain a large “teacher” vision model that learns rich visual representations, then **distill** those representations into a smaller **student backbone** that you can deploy and reuse across downstream tasks such as world model training, Vision-Language-Action (VLA) policies, detection, or segmentation.

At small scale, teacher–student distillation can look like a simple training script. At realistic scale, such as with large image datasets, multiple GPUs, or long-running jobs, it becomes a **distributed systems problem**, not just a modeling exercise.

In practice, you must:
- read and decode large image datasets efficiently,
- run CPU-side preprocessing without stalling GPUs,
- execute frozen teacher forward passes at scale,
- synchronize student gradients across GPUs,
- and checkpoint progress so long-running jobs recover cleanly from failures.

In this notebook, you learn how to **implement and scale the teacher–student distillation pattern** using:
- **Ray Data** for distributed, CPU-side image loading and preprocessing, and
- **Ray Train** for multi-GPU distributed training with checkpointing and resume.

You use a pretrained ImageNet model as a stand-in for a foundation-model teacher and distill it into a smaller student backbone. The same infrastructure pattern applies directly to self-supervised teachers (for example DINO or MAE) trained on large unlabeled robotics datasets.

---

## What you will learn

By the end of this notebook, you will be able to:

- Load and shard an image dataset across a cluster using **Ray Data**
- Perform scalable, CPU-side image preprocessing without blocking GPUs
- Implement **teacher–student feature distillation** with a frozen teacher encoder
- Train a student model using **Ray Train** with multi-GPU Distributed Data Parallel (DDP)
- Add **checkpointing and fault tolerance** so distillation jobs recover from failures
- Export a distilled backbone for reuse in downstream robotics tasks (world models, VLA, etc.)

---

## What you are optimizing

In teacher–student distillation, you train a **student encoder** to match the representation produced by a frozen **teacher**.

Let:
- $f_T(x)$ denote the teacher embedding for an input image $x$,
- $f_S(x)$ denote the student embedding,
- $P(\cdot)$ denote a learnable projection that maps student features into the teacher feature space,
- $\mathcal{D}$ denote the training dataset.

You optimize the student by minimizing a representation-matching objective:

$$
\mathcal{L}_{\text{distill}}
\;=\;
\mathbb{E}_{x \sim \mathcal{D}}
\left[
\left\lVert
\operatorname{norm}\!\left(P\!\left(f_S(x)\right)\right)
-
\operatorname{norm}\!\left(f_T(x)\right)
\right\rVert_2^2
\right],
$$

where $\operatorname{norm}(\cdot)$ denotes feature normalization (for example, $\ell_2$ normalization).

To make the pipeline end-to-end measurable in this notebook, you also attach a small supervised head to the student and optimize a composite objective:

$$
\mathcal{L}
\;=\;
\mathcal{L}_{\text{distill}}
\;+\;
\lambda\,\mathcal{L}_{\text{sup}},
$$

where:
- $\mathcal{L}_{\text{sup}}$ is a standard supervised loss (cross-entropy in this notebook),
- $\lambda$ controls the relative weight of supervised learning.

In practical robotics workflows, you often set $\lambda = 0$ and distill on **unlabeled fleet images**, then attach task-specific heads (BEV, action, detection) in later training stages.


---

## Why this objective drives the system design

The mathematical objective is simple. Evaluating it efficiently at scale is not.

To compute this loss across a cluster, you must:
- preprocess images on CPUs without blocking GPUs,
- stream fixed-shape tensors efficiently to multiple workers,
- run frozen teacher forward passes at high throughput,
- synchronize gradients and metrics across GPUs,
- and checkpoint model state so training recovers from interruptions.

The remainder of this notebook shows how **Ray Data** and **Ray Train** help you evaluate this objective **reliably and efficiently on a cluster**.

---

## Why use Ray Data and Ray Train for teacher–student training?

Teacher–student distillation is a well-established pattern in the robotics and vision community. Many teams use specialized tools that implement this workflow end-to-end, particularly for large-scale self-supervised learning (SSL) on unlabeled image corpora.

For example, **LightlyTrain** provides an opinionated framework for:
- self-supervised pretraining (e.g., DINO-style objectives),
- teacher–student distillation,
- and dataset sampling and management.

If your primary goal is to **run SSL and distillation as a self-contained workflow**, tools like LightlyTrain are often an excellent choice.

However, these frameworks typically **own the entire training loop and execution model**. As a result, they do not naturally integrate with:
- Ray Data for distributed, heterogeneous preprocessing,
- Ray Train for cluster-wide execution, fault tolerance, and checkpoint orchestration,
- or larger robotics pipelines that mix multiple training stages, models, and workloads.

In contrast, this notebook focuses on **the infrastructure layer**, not on reimplementing SSL algorithms.

You use **Ray Data + Ray Train** here because they allow you to:
- scale teacher–student training across CPUs and GPUs using a unified execution model,
- integrate distillation into broader robotics workflows (BEV training, VLA finetuning, multimodal fusion),
- mix and match training stages (SSL pretraining, distillation, downstream task finetuning) without switching orchestration frameworks,
- and productionize long-running jobs with checkpointing, retries, and observability.

In other words, Ray does not replace specialized SSL frameworks. Instead, it provides a **general, composable foundation** for running teacher–student training *as part of a larger robotics system*.

This notebook demonstrates how to implement the **teacher–student pattern itself** using Ray primitives so the same execution model can support:
- large-scale SSL pretraining (with external or custom objectives),
- representation distillation,
- and downstream robotics tasks such as BEV perception or Vision-Language-Action policies.

---

## Who this notebook is for

This notebook is for **technical practitioners in robotics**, including:
- ML engineers building reusable vision backbones,
- research engineers scaling representation learning beyond a single node,
- teams evaluating whether **Ray and Anyscale** simplify training infrastructure.

You should be comfortable with Python and PyTorch. You do not need prior experience with Ray.

The goal is to help you build **repeatable, scalable teacher–student pipelines on real infrastructure**, not just to run a single distillation experiment.

---

## How to interpret the scale of this notebook

This notebook runs in a **multi-node, multi-GPU configuration** by default. It launches **two Ray Train workers**, each with access to a single GPU, and distributes preprocessing and training across the cluster.

The defaults are intentionally conservative, using small batch sizes, limited epochs, and a compact dataset, so you can iterate quickly on modest hardware (for example A10 GPUs). You scale the same code path to larger datasets, larger teachers, and more GPUs by adjusting a small number of parameters.

The emphasis is on demonstrating a **scalable distributed training pattern** that you can reuse across robotics workloads.

---

## Where this fits in the course

Earlier in this course you fine-tuned a 3.4B-parameter VLA policy (notebook 02) and pre-trained a V-JEPA world model (notebook 04). Both are powerful -- and both are far too large to run on a robot's onboard compute. In practice you deploy a **small** model, and the standard way to get one without discarding what the large model learned is **teacher-student distillation**: train a compact student to reproduce a large teacher's representations.

This notebook covers that final, deployment-facing step. The teacher here is a frozen ImageNet vision backbone -- a stand-in for any large encoder, including the V-JEPA vision tower from notebook 04 or a VLA's perception stack -- and you distill it into a small student backbone you could ship to the edge. It closes the course's arc: **scale up to learn (notebooks 02, 04), scale down to deploy (notebook 05)** -- all on the same Ray Data + Ray Train stack.

## Cell 1: Verify runtime and connect to Ray

You confirm your Python, PyTorch, TorchVision, and Ray runtime versions and then connect to a Ray cluster (or start a local Ray runtime if none is available). This helps you catch environment mismatches early and ensures Ray can accept distributed jobs.

**What you do**
- Print core version strings so you can debug dependency issues quickly.
- Try to connect to an existing Ray cluster with `address="auto"`. If that fails, start a local Ray runtime.
- Print cluster resources so you can confirm available CPUs/GPUs and memory.

**What to check**
- You should see printed versions for Python, Torch, TorchVision, and Ray.
- You should see either `Connected to Ray cluster via address='auto'.` or `Started local Ray runtime.`
- `Cluster resources:` should list CPU and GPU counts (or other cluster resources).

**Why it matters**
- You ensure the notebook runs against the intended Ray deployment and that your environment is compatible with the distributed training you plan to run.

In [ ]:
# Cell 1: Environment check + Ray initialization
# You import core libs, print versions for quick debugging, and connect to Ray.
import os, sys, time, math, random, tempfile
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms

import ray
import ray.data
import ray.train as train
from ray.train import (
    Checkpoint,
    DataConfig,
    RunConfig,
    ScalingConfig,
    FailureConfig,
    CheckpointConfig,
)
from ray.train.torch import TorchTrainer

# Print runtime versions so you can quickly detect mismatches between local
# and cluster environments (Python, PyTorch, TorchVision, Ray).
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Ray:", ray.__version__)

# --- Ray init (Anyscale notebook/workspace safe) ---
# You prefer to connect to an existing cluster first (address="auto").
# If that fails (for example, running locally or inside a notebook without
# an attached cluster), you fall back to starting a local Ray runtime.
if ray.is_initialized():
    print("Ray is already initialized.")
else:
    try:
        ray.init(address="auto")
        print("Connected to Ray cluster via address='auto'.")
    except Exception:
        ray.init()
        print("Started local Ray runtime.")

# Print cluster resources so you can validate the number of CPUs/GPUs and available memory.
# This helps you choose sensible scaling configs later.
print("Cluster resources:", ray.cluster_resources())


## Cell 2: Load and shard an image dataset with Ray Data

You load a small image classification dataset from Hugging Face and convert it into **Ray Data datasets** that can be processed and sharded across a cluster.

You intentionally use **CIFAR-10** in this notebook because it is:
- small enough to run quickly end to end,
- representative of image-based robotics workloads,
- and simple to reason about when validating the distillation pipeline.

**What you do**
- Load a Hugging Face dataset via `load_dataset`, with an overrideable `DATASET_ID`.
- Convert Hugging Face splits into Ray Data datasets using `ray.data.from_huggingface(...)`.
- Split the training set into train/validation deterministically.
- Optionally limit the number of rows in each split so the tutorial runs quickly.
- Inspect dataset size and schema.

**What to check**
- You should see printed row counts for Train, Val, and Test that respect the `MAX_*_ROWS` limits.
- The dataset schema should include an image column (for example `image`) and a label column.
- `Sample row keys` should show the available fields that downstream preprocessing expects.

**Why it matters**
- You convert an external dataset into Ray’s distributed data abstraction early, so all later preprocessing, sharding, and streaming happens at cluster scale without rewriting your training loop.

In [ ]:
# Cell 2: Dataset loading and Ray Data conversion
# You load a small image dataset and convert it into Ray Data datasets that
# can be split, limited, and sharded across a cluster.
from datasets import load_dataset

# Hugging Face CIFAR-10 dataset (columns: "image", "label").
# You intentionally choose a small, well-known dataset so the teacher–student
# distillation pattern is easy to run end to end in a tutorial.
DATASET_ID = os.environ.get("DATASET_ID", "uoft-cs/cifar10")

# Load the dataset via Hugging Face Datasets.
hf = load_dataset(DATASET_ID)

# Convert Hugging Face splits into Ray Data datasets.
# This lets you apply distributed preprocessing and stream data efficiently
# into Ray Train workers later.
train_ds = ray.data.from_huggingface(hf["train"])
test_ds = ray.data.from_huggingface(hf["test"])

# Split the training dataset into train/validation sets deterministically.
# You keep the split fixed so results are reproducible across runs.
train_ds, val_ds = train_ds.train_test_split(test_size=0.1, seed=42)

# Optional row limits to keep the tutorial fast.
# You can override these via environment variables for larger runs.
MAX_TRAIN_ROWS = int(os.environ.get("MAX_TRAIN_ROWS", "20000"))
MAX_VAL_ROWS = int(os.environ.get("MAX_VAL_ROWS", "2000"))
MAX_TEST_ROWS = int(os.environ.get("MAX_TEST_ROWS", "2000"))

# Apply row limits to each split.
train_ds = train_ds.limit(MAX_TRAIN_ROWS)
val_ds = val_ds.limit(MAX_VAL_ROWS)
test_ds = test_ds.limit(MAX_TEST_ROWS)

# Materialize counts so you can sanity-check dataset sizes.
# Note: count() triggers execution, which is fine for this small tutorial dataset. In practice do note perform this step
print("Train rows:", train_ds.count())
print("Val rows:", val_ds.count())
print("Test rows:", test_ds.count())

# Inspect schema and sample keys to confirm expected columns are present.
# Downstream preprocessing relies on an image column and a label column.
print("Train schema:", train_ds.schema())
print("Sample row keys:", train_ds.take(1)[0].keys())

## Cell 3: Define image preprocessing and Ray Data → Torch collation

You define the **image preprocessing pipeline** and the **collate functions** that convert Ray Data batches into model-ready PyTorch tensors.

You standardize all images to a fixed resolution and normalization so:
- the teacher and student networks see consistent inputs,
- CPU-side preprocessing stays deterministic and scalable,
- and GPU workers receive fixed-shape tensors.

**What you do**
- Define a configurable input resolution via `IMAGE_SIZE`.
- Specify ImageNet normalization statistics to match pretrained vision backbones.
- Build separate transform pipelines for training (with augmentation) and evaluation (deterministic).
- Handle multiple image encodings (PIL, Hugging Face image dicts, NumPy arrays).
- Create Ray Data–compatible collate functions that apply transforms and return Torch tensors.

**What to check**
- `train_collate` and `eval_collate` exist and accept Ray Data batches.
- Each batch returns:
  - `image`: a tensor of shape `(B, 3, IMAGE_SIZE, IMAGE_SIZE)` with `float32` values.
  - `label`: a tensor of shape `(B,)` with `int64` class indices.
- No image-type errors occur when iterating over Ray Data.

**Why it matters**
- You move all image decoding and augmentation into a clean CPU-side boundary, which lets Ray Data scale preprocessing independently of GPU training.

In [ ]:
# Cell 3: Image preprocessing and Ray Data collation
# You define how raw dataset images are converted into normalized torch tensors
# that both the teacher and student networks can consume.
IMAGE_SIZE = int(os.environ.get("IMAGE_SIZE", "224"))

# ImageNet normalization statistics.
# You use these to stay compatible with pretrained torchvision backbones.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

from PIL import Image
import io
import numpy as np

# Training-time transforms.
# You include random crops and flips to introduce basic data augmentation.
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Evaluation-time transforms.
# You keep these deterministic so validation metrics are stable and comparable.
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def _to_pil(img):
    """
    Convert a dataset image into a PIL Image.

    You handle multiple input formats so the collate function works
    regardless of how the dataset encodes images.
    """
    # Case 1: image is already a PIL Image.
    if hasattr(img, "convert"):
        return img

    # Case 2: Hugging Face Image dict (bytes or filesystem path).
    if isinstance(img, dict):
        if img.get("bytes") is not None:
            return Image.open(io.BytesIO(img["bytes"]))
        if img.get("path") is not None:
            return Image.open(img["path"])

    # Case 3: NumPy array (H, W, C).
    if isinstance(img, np.ndarray):
        return Image.fromarray(img)

    # Anything else is unexpected and should fail loudly.
    raise TypeError(f"Unsupported image type: {type(img)}")

def make_collate_fn(transform):
    """
    Build a Ray Data-compatible collate function.

    You apply the given transform to each image and return
    torch tensors ready for GPU training.
    """
    def collate(batch):
        # Support common image field names ("image" or "img").
        imgs = batch.get("image", batch.get("img"))
        labels = batch.get("label")

        # Convert images to PIL, apply transforms, and stack into a batch tensor.
        x = torch.stack([transform(_to_pil(im).convert("RGB")) for im in imgs])

        # Convert labels into a LongTensor for classification losses.
        y = torch.tensor(labels, dtype=torch.long)
        return {"image": x, "label": y}
    return collate

# Collate functions used by Ray Data when streaming batches
# into the training and evaluation loops.
train_collate = make_collate_fn(train_tf)
eval_collate  = make_collate_fn(eval_tf)

## Cell 4: Build teacher and student models (infer feature dims at runtime)

You construct a frozen pretrained **teacher encoder** and a flexible **student backbone** that infers its output feature dimensionality at runtime. You then wrap the student with a small **projection head** (to map student features into the teacher space) and a toy **classification head** so you can measure a downstream objective alongside distillation.

**What you do**
- Create a frozen ResNet-50 teacher whose final classification layer is removed so forward returns pooled features `(B, 2048)`.
- Create a MobileNetV3-Large student backbone and strip its classifier so forward returns backbone features; infer the student feature dimension using a dummy forward.
- Build `StudentNet` that returns `(feats, proj_feats, logits)` where:
  - `feats`: raw student features (flattened),
  - `proj_feats`: linear projection of `feats` into the teacher feature space,
  - `logits`: toy downstream classifier outputs.

**What to check**
- Teacher returns embeddings of size `teacher_dim == 2048`.
- Student `feat_dim` is inferred automatically and matches the projection input.
- `StudentNet.forward(x)` returns three tensors with expected shapes:
  - `feats.shape == (B, student_dim)`,
  - `proj_feats.shape == (B, teacher_dim)`,
  - `logits.shape == (B, num_classes)`.

**Why it matters**
- You infer student dims dynamically so the distillation projection aligns with the teacher regardless of torchvision version or slight implementation differences in backbones. This reduces brittle assumptions and lets you swap backbones easily.

In [ ]:
# Cell 4: Robust model definitions with runtime feature-dim inference
# You build a frozen ResNet-50 teacher and a MobileNetV3 student whose output
# dimensionality you infer with a dummy forward so the projection head matches.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights, mobilenet_v3_large, MobileNet_V3_Large_Weights

def build_teacher_encoder(device=torch.device("cpu")):
    """
    Build a frozen ResNet-50 teacher.

    You replace the final FC head with Identity so forward returns pooled features
    with shape (B, 2048), and you freeze all parameters to avoid accidental updates.
    """
    teacher = resnet50(weights=ResNet50_Weights.DEFAULT)

    # Replace the classification head so .forward() returns pooled features (B, 2048).
    teacher.fc = nn.Identity()
    teacher.eval()

    # Freeze teacher parameters: distillation must not update the teacher.
    for p in teacher.parameters():
        p.requires_grad = False

    teacher.to(device)
    teacher_dim = 2048  # ResNet-50 pooled feature size
    return teacher, teacher_dim

def build_student_backbone(device=torch.device("cpu")):
    """
    Build a MobileNetV3-Large student backbone and infer its output dimension.

    Different torchvision versions expose different attributes (e.g., `.classifier`).
    You replace or remove those heads so a forward pass yields raw backbone features.
    Then you run a dummy forward to compute the flattened feature dimension.
    """
    backbone = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

    # Many torchvision models end with `.classifier`; replace with Identity so
    # forward returns the penultimate feature tensor rather than logits.
    if hasattr(backbone, "classifier"):
        backbone.classifier = nn.Identity()

    backbone.to(device)
    backbone.eval()

    # Infer output shape via a dummy forward (avoids hard-coding student dims).
    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        out = backbone(dummy)
    feat_dim = out.shape[-1] if out.ndim == 2 else int(torch.prod(torch.tensor(out.shape[1:])))

    # If output is (1, N) treat N as feat_dim; if output is multi-dim (e.g., (1, C, H, W)),
    # flatten to (1, -1) and take product.
    if isinstance(out, torch.Tensor) and out.ndim > 2:
        feat_dim = int(out.flatten(1).shape[1])
    return backbone, feat_dim

class StudentNet(nn.Module):
    """
    Student backbone + projection head + toy classifier.

    Forward returns:
      - feats:   flattened backbone features (B, student_dim)
      - proj_feats: linear projection (B, teacher_dim) used for distillation
      - logits:  classifier outputs for a toy supervised objective (B, num_classes)
    """
    def __init__(self, num_classes: int, teacher_dim: int, device=torch.device("cpu")):
        super().__init__()
        # Build backbone and capture inferred student feature dim
        self.backbone, student_dim = build_student_backbone(device=device)
        self.student_dim = student_dim

        # Projection maps student features into teacher feature space for distillation.
        self.proj = nn.Linear(self.student_dim, teacher_dim)

        # Toy downstream head (keeps demo end-to-end measurable).
        self.classifier = nn.Linear(self.student_dim, num_classes)

    def forward(self, x):
        # Obtain backbone features. Could be (B, student_dim) or (B, C, H, W).
        feats = self.backbone(x)    

        # Flatten spatial dims if present so downstream heads receive (B, student_dim).     
        if feats.ndim > 2:
            feats = feats.flatten(1)

        # Project into teacher space and compute logits for classification.
        proj_feats = self.proj(feats)    
        logits = self.classifier(feats)  
        return feats, proj_feats, logits

def build_models_for_training(device=torch.device("cuda" if torch.cuda.is_available() else "cpu")):
    """
    Build teacher and student models ready for training/evaluation on `device`.

    You return (teacher, student, teacher_dim). The student is not wrapped for DDP here;
    that happens later inside the Ray Train worker (via prepare_model).
    """
    teacher, teacher_dim = build_teacher_encoder(device=device)
    # Create a student with inferred feature dims so projection aligns automatically.
    student = StudentNet(num_classes=10, teacher_dim=teacher_dim, device=device)
    return teacher, student, teacher_dim


## Cell 5: Distributed training loop — aggregation, training, validation, and checkpointing

You implement the per-worker training loop that Ray Train runs on every GPU worker.  
This cell does four critical things:

- You aggregate scalar metrics across workers with all-reduce helpers so reported values reflect the whole distributed job.  
- You build teacher and student models, enable AMP, and optionally resume from a checkpoint.  
- You stream batches from Ray Data shards, compute the distillation + supervised losses, and run optimizer steps with gradient scaling.  
- You evaluate on a validation shard, reduce accuracy across workers, and let rank 0 write checkpoints and report them to Ray Train.

**What to check**
- Training and validation metrics (`train/loss`, `train/distill`, `val/acc`) appear in the run logs and the metrics DataFrame.
- Checkpoint files (`state.pt`) are created by rank 0 and attached to reported metrics.
- The all-reduce helpers return sensible global aggregates when you run with >1 worker.

**Why it matters**
- You keep GPU work focused on forward/backward passes while Ray Data handles CPU-side I/O and transforms.
- You make long-running distillation jobs robust to worker failures and restarts by checkpointing and resume.

In [ ]:
# Cell 5: Distributed helpers + per-worker train loop 
# You define helper functions for cross-worker aggregation and a full train loop
# that Ray Train will execute on each worker. Comments explain intent at each step.

def _all_reduce_mean(value: float, device: torch.device) -> float:
    """All-reduce a scalar mean across workers (safe for num_workers=1)."""
    # If torch.distributed is initialized, sum the scalar across ranks and divide
    # by world size so every worker obtains the global mean.
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        t = torch.tensor(value, device=device, dtype=torch.float32)
        torch.distributed.all_reduce(t, op=torch.distributed.ReduceOp.SUM)
        t /= train.get_context().get_world_size()
        return t.item()
    # If distributed is not active, return the value unchanged (single-worker case).
    return float(value)

def _all_reduce_sum_pair(a: float, b: float, device: torch.device):
    """All-reduce (sum) a pair of scalars across workers."""
    # Use a two-element tensor to sum both scalars in a single collective call.
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        t = torch.tensor([a, b], device=device, dtype=torch.float64)
        torch.distributed.all_reduce(t, op=torch.distributed.ReduceOp.SUM)
        return float(t[0].item()), float(t[1].item())
    # Single-worker fallback returns the original values.
    return float(a), float(b)


def train_loop_per_worker(config: dict):
    # Local imports so the function is self-contained when Ray serializes it.
    import os, time, tempfile
    import numpy as np
    import random

    import torch
    import torch.nn.functional as F

    import ray.train as train
    from ray.train import Checkpoint
    from ray.train.torch import prepare_model, get_device


    # Determinism (best-effort)
    # You derive a per-worker seed from the base seed plus the world rank so that
    # data shuffling / augmentations differ across workers but remain reproducible.
    base_seed = int(config.get("seed", 42))
    world_rank = train.get_context().get_world_rank()
    seed = base_seed + world_rank
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Use CuDNN benchmark to potentially improve throughput for fixed-size inputs.
    torch.backends.cudnn.benchmark = True

    # Get the device assigned to this worker (Ray sets CUDA_VISIBLE_DEVICES per worker).
    device = get_device()

    # --- Build teacher + student ---
    # Build and freeze the teacher; move it to the worker device and set eval mode.
    teacher, teacher_dim = build_teacher_encoder()
    teacher.to(device)
    teacher.eval()

    # Build student and wrap it for DDP via Ray Train's prepare_model.
    student = StudentNet(num_classes=config["num_classes"], teacher_dim=teacher_dim).to(device)
    student = prepare_model(student)

    # Optimizer for student parameters (projection + backbone + classifier).
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=float(config["lr"]),
        weight_decay=float(config.get("weight_decay", 0.05)),
    )

    # Mixed precision: enable GradScaler if AMP requested.
    amp_enabled = bool(config.get("amp", True))
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    # --- Resume if a checkpoint exists ---
    # If Ray Train provides a checkpoint (from a previous run), load student/optimizer/scaler state.
    checkpoint = train.get_checkpoint()
    start_epoch = 0
    if checkpoint is not None:
        with checkpoint.as_directory() as checkpoint_dir:
            state = torch.load(os.path.join(checkpoint_dir, "state.pt"), map_location="cpu")
        model_ref = student.module if hasattr(student, "module") else student
        model_ref.load_state_dict(state["student"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = int(state["epoch"]) + 1

    # --- Ray Data shards ---
    # Retrieve the named dataset shards that TorchTrainer attached ("train" and "val").
    train_shard = train.get_dataset_shard("train")
    val_shard = train.get_dataset_shard("val")

    # Epoch loop (supports resume from checkpoint via start_epoch).
    for epoch in range(start_epoch, int(config["num_epochs"])):
        # ----------------
        # Train
        # ----------------
        student.train()

        t0 = time.time()
        running_loss = 0.0
        running_ce = 0.0
        running_distill = 0.0
        num_batches = 0
        num_samples = 0

        # Ask Ray Data to yield PyTorch batches placed on the worker device.
        # You supply the collate function that converts raw rows -> tensors.
        train_iter = train_shard.iter_torch_batches(
            batch_size=int(config["batch_size"]),
            collate_fn=train_collate,
            prefetch_batches=int(config.get("prefetch_batches", 2)),
        )

        # Iterate over mini-batches streamed from the dataset shard.
        for batch in train_iter:
            # Move data to device (non_blocking when pinned memory is available).
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Mixed precision forward/backward context.
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                # Run frozen teacher forward pass under no_grad to avoid gradient compute.
                with torch.no_grad():
                    teacher_feats = teacher(images)  # (B, teacher_dim)

                # Student forward returns raw feats, projected feats for distillation, and logits.
                _, student_proj, logits = student(images)

                # Feature distillation: normalize both features and use cosine distance.
                t_feat = F.normalize(teacher_feats, dim=-1)
                s_feat = F.normalize(student_proj, dim=-1)
                distill_loss = (1.0 - (t_feat * s_feat).sum(dim=-1)).mean()

                # Toy supervised objective (keeps the tutorial measurable end-to-end).
                ce_loss = F.cross_entropy(logits, labels)

                # Composite loss: supervised + weighted distillation term.
                loss = ce_loss + float(config["distill_weight"]) * distill_loss

            # Backpropagate using the scaler and update optimizer.
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Accumulate scalar stats on this worker for later reduction.
            running_loss += float(loss.detach().cpu())
            running_ce += float(ce_loss.detach().cpu())
            running_distill += float(distill_loss.detach().cpu())
            num_batches += 1
            num_samples += int(images.shape[0])

        # Compute per-worker averages and throughput.
        train_loss = running_loss / max(num_batches, 1)
        train_ce = running_ce / max(num_batches, 1)
        train_distill = running_distill / max(num_batches, 1)
        imgs_per_s = num_samples / max(time.time() - t0, 1e-9)

        # Aggregate scalar metrics across workers so reported numbers reflect the whole job.
        train_loss = _all_reduce_mean(train_loss, device)
        train_ce = _all_reduce_mean(train_ce, device)
        train_distill = _all_reduce_mean(train_distill, device)
        imgs_per_s = _all_reduce_mean(imgs_per_s, device)

        # ----------------
        # Validate
        # ----------------
        student.eval()
        val_correct = 0.0
        val_total = 0.0
        val_loss_sum = 0.0
        val_batches = 0

        # Create a validation iterator (device-placed batches) using the eval collate fn.
        val_iter = val_shard.iter_torch_batches(
            batch_size=int(config["eval_batch_size"]),
            collate_fn=eval_collate,
            prefetch_batches=int(config.get("prefetch_batches", 2)),
        )

        # Run through validation without gradient updates.
        with torch.no_grad():
            for batch in val_iter:
                images = batch["image"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=amp_enabled):
                    _, _, logits = student(images)
                    vloss = F.cross_entropy(logits, labels)

                preds = logits.argmax(dim=-1)
                val_correct += float((preds == labels).sum().item())
                val_total += float(labels.numel())
                val_loss_sum += float(vloss.detach().cpu())
                val_batches += 1

        # Reduce validation counts across workers (sum) and compute global accuracy.
        val_correct, val_total = _all_reduce_sum_pair(val_correct, val_total, device)
        val_acc = val_correct / max(val_total, 1.0)
        val_loss = _all_reduce_mean(val_loss_sum / max(val_batches, 1), device)

        # ----------------
        # Report + checkpoint
        # ----------------
        metrics = {
            "epoch": epoch,
            "train/loss": train_loss,
            "train/ce": train_ce,
            "train/distill": train_distill,
            "train/imgs_per_s": imgs_per_s,
            "val/loss": val_loss,
            "val/acc": val_acc,
        }

        # Best practice: only rank 0 writes a checkpoint to avoid races and duplicate uploads.
        checkpoint_to_report = None
        if train.get_context().get_world_rank() == 0:
            # Save model and optimizer/scaler state to a temporary directory and
            # convert it into a Ray Train Checkpoint that the system will persist.
            with tempfile.TemporaryDirectory() as tmpdir:
                model_ref = student.module if hasattr(student, "module") else student
                torch.save(
                    {
                        "epoch": epoch,
                        "student": model_ref.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scaler": scaler.state_dict(),
                        "config": config,
                    },
                    os.path.join(tmpdir, "state.pt"),
                )
                checkpoint_to_report = Checkpoint.from_directory(tmpdir)
                # Report metrics and attach the checkpoint so Ray persists it.
                train.report(metrics=metrics, checkpoint=checkpoint_to_report)
        else:
            # Non-zero ranks report metrics but do not attach a checkpoint.
            train.report(metrics=metrics, checkpoint=None)


## Cell 6: Configure and launch the TorchTrainer run

You configure the experiment, choose storage and scaling settings, and launch a distributed TorchTrainer job that runs the per-worker training loop you defined earlier.

**What you do**
- You collect hyperparameters and runtime toggles into a `config` dict (seed, LR, batch sizes, distillation weight, AMP).
- You pick a stable `storage_path` that works on Anyscale if available, otherwise you fall back to a local path.
- You create `RunConfig` with a name, storage location, failure retry policy, and checkpoint retention/selection rules.
- You define a `ScalingConfig` that determines how many workers and whether each uses a GPU.
- You instruct TorchTrainer to split the `train` and `val` Ray Data datasets evenly across workers via `DataConfig`.
- You instantiate `TorchTrainer` with the per-worker loop, configs, and datasets, then call `trainer.fit()` to run the job.

**What to check**
- The run starts and allocates the expected number of workers (check Ray dashboard or `trainer` logs).
- Checkpoints are written to `storage_path` and ranked/kept according to `CheckpointConfig`.
- Metrics (train/val) appear in `result.metrics_dataframe` and `result.path` points to the persisted run.

**Why it matters**
- You centralize run configuration so you can reproduce experiments and scale by changing a few parameters.
- You persist checkpoints and metrics in shared storage so long-running experiments recover from failures and you can inspect results after the job finishes.

In [ ]:
# Cell 6: Trainer configuration, scaling, and fit launch
# You assemble runtime and hyperparameter settings, choose a storage path,
# configure checkpointing and scaling, and then launch TorchTrainer.

# --- Experiment hyperparameters and runtime config ---
config = {
    "seed": 42,
    "num_classes": 10,
    "num_epochs": int(os.environ.get("NUM_EPOCHS", "3")),
    "batch_size": int(os.environ.get("BATCH_SIZE", "64")),
    "eval_batch_size": int(os.environ.get("EVAL_BATCH_SIZE", "128")),
    "lr": float(os.environ.get("LR", "3e-4")),
    "weight_decay": float(os.environ.get("WEIGHT_DECAY", "0.05")),
    "distill_weight": float(os.environ.get("DISTILL_WEIGHT", "1.0")),
    "amp": True,
    "prefetch_batches": int(os.environ.get("PREFETCH_BATCHES", "2")),
}

# --- Storage path: prefer shared cluster storage when available ---
# You choose /mnt/cluster_storage when running on Anyscale so checkpoints
# and logs land on a cluster-visible filesystem. Otherwise, you use a home dir.
if os.path.exists("/mnt/cluster_storage"):
    storage_path = os.path.join("/mnt/cluster_storage", "ray_results")
else:
    storage_path = os.path.expanduser("~/ray_results")

# Experiment / run name (override with RAY_RUN_NAME env var if desired).
exp_name = os.environ.get("RAY_RUN_NAME", "teacher_student_distillation")

# --- RunConfig: persistence and failure handling ---
# You configure how many failures to tolerate and how checkpoints are kept/scored.
run_config = RunConfig(
    name=exp_name,
    storage_path=storage_path,
    failure_config=FailureConfig(max_failures=int(os.environ.get("MAX_FAILURES", "2"))),
    checkpoint_config=CheckpointConfig(
        num_to_keep=int(os.environ.get("NUM_TO_KEEP", "3")),
        checkpoint_score_attribute="val/acc",
        checkpoint_score_order="max",
    ),
)

# --- ScalingConfig: how many workers and whether to use GPUs ---
# You control parallelism via NUM_WORKERS. Each worker will run the per-worker loop.
scaling_config = ScalingConfig(
    num_workers=int(os.environ.get("NUM_WORKERS", "2")),
    use_gpu=True,
)

# --- Dataset split behavior ---
# You instruct Ray Train to split the named datasets ("train" and "val")
# across workers so each worker gets its own shard to iterate.
dataset_config = DataConfig(datasets_to_split=["train", "val"])

# --- Instantiate TorchTrainer ---
# You pass the per-worker function, config, scaling, run, and datasets to the trainer.
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={
        "train": train_ds,
        "val": val_ds,
    },
    dataset_config=dataset_config,
)

# --- Launch the distributed training job ---
# .fit() blocks until the training job finishes (or errors / is retried per failure_config).
result = trainer.fit()
print("Training finished.")
print("Run path:", result.path)


## Cell 7: Inspect training metrics after the run completes

You load the metrics recorded during training into a DataFrame and inspect the most recent entries.

Ray Train automatically persists metrics that you report from the training loop (including those attached to checkpoints). After the run finishes, you can access all of them directly from the `Result` object.

**What you do**
- Retrieve the full metrics table via `result.metrics_dataframe`.
- Inspect the last few rows to see how metrics evolved over the final epochs.

**What to check**
- The DataFrame should include columns such as:
  - `train/loss`, `train/distill`, `train/ce`
  - `val/loss`, `val/acc`
  - `epoch`
- Metric values should change across epochs and reflect convergence trends.
- The final rows should correspond to the last completed epoch.

**Why it matters**
- You get a simple, programmatic view of training progress without manually logging metrics.
- You can use this table to debug, visualize learning curves, or select checkpoints based on validation performance.

In [ ]:
# Cell 7: Inspect persisted training metrics
# You access the metrics DataFrame that Ray Train collected and stored
# during the distributed training run.

# Metrics reported via train.report(...) are aggregated and persisted by Ray Train.
df = result.metrics_dataframe

# Display the last few metric entries so you can quickly inspect
# final training and validation behavior.
df.tail(10)


In [ ]:
best_ckpt = result.get_best_checkpoint(metric="val/acc", mode="max")
print("Best checkpoint:", best_ckpt)

if best_ckpt is None:
    # Fall back to the last checkpoint (if present)
    best_ckpt = result.checkpoint
    print("Fallback to last checkpoint:", best_ckpt)


## Cell 8: Select the best checkpoint by validation accuracy

You select the checkpoint with the highest validation accuracy from the completed training run. If no ranked checkpoint is available, you fall back to the most recent checkpoint.

Ray Train tracks checkpoints and their associated metrics automatically, which lets you retrieve the “best” model without manual bookkeeping.

**What you do**
- Query the run for the checkpoint that maximizes `val/acc`.
- Print the selected checkpoint so you can confirm it exists.
- Fall back to the last checkpoint if no scored checkpoint is found.

**What to check**
- `Best checkpoint:` should print a valid `Checkpoint` object.
- If no best checkpoint is found, the fallback path should print a non-`None` checkpoint.
- The selected checkpoint should correspond to the highest observed validation accuracy.

**Why it matters**
- You decouple model selection from the training loop.
- You make it easy to evaluate, export, or reuse the strongest student model produced by the run.

In [ ]:
# --- Load checkpoint into a local model and evaluate on test set ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Eval device:", device)

# Teacher dim is ResNet50 embedding dim (fixed here)
teacher_dim = 2048

student_eval = StudentNet(num_classes=10, teacher_dim=teacher_dim).to(device)
student_eval.eval()

assert best_ckpt is not None, "No checkpoint found to evaluate."
with best_ckpt.as_directory() as ckpt_dir:
    state = torch.load(os.path.join(ckpt_dir, "state.pt"), map_location="cpu")
student_eval.load_state_dict(state["student"])

correct = 0
total = 0
test_iter = test_ds.iter_torch_batches(
    batch_size=256,
    collate_fn=eval_collate,
    prefetch_batches=2,
)

with torch.no_grad():
    for batch in test_iter:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        _, _, logits = student_eval(images)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(labels.numel())

print(f"Test accuracy (toy downstream head): {correct/total:.4f} ({correct}/{total})")


---

## That's the loop

Across this course you built an end-to-end physical-AI workflow on Ray + Anyscale:

- **01** streamed robotics video with Ray Data
- **02** fine-tuned a 3.4B VLA with Ray Train
- **03** served it with Ray Serve, evaluated it in Isaac Lab via Ray tasks, and closed the loop with `union()`
- **04** pre-trained a V-JEPA world model at scale -- a learned simulator
- **05** distilled a large encoder into an edge-deployable student

The through-line: the **same** Ray Data + Ray Train + Ray Serve primitives carried every stage, from a 3.4B model on many GPUs down to a backbone that fits on a robot. Change the config, not the code.